In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit

# from pathlib import Path
import joblib

In [2]:

# Load dataset
DATA_PATH = Path("../data/processed/terai_features_engineered.parquet")
df = pd.read_parquet(DATA_PATH)
df = df.sort_values(by=["Date", "District"]).reset_index(drop=True)
df.head(3)


,Date,Year,Day_of_Year,Province,District,Regional_Hub,Latitude,Longitude,Max_Temp_C,Min_Temp_C,...,DOY_Sin,DOY_Cos,Month_Sin,Month_Cos,Season,Rain_Log,Yesterday_Rain_Log,Past_7D_Rain_Log,Is_Western_Terai,Terai_Region
0,2000-01-15,2000,15,Lumbini,Banke,Nepalgunj,28.05,81.62,25.11,10.84,...,0.2552,0.9669,0.5,0.866,Winter,0.0000,0.00,2.3915,1,Western Terai
1,2000-01-15,2000,15,Madhesh,Bara,Kalaiya,27.03,85.00,27.77,12.74,...,0.2552,0.9669,0.5,0.866,Winter,0.0392,0.01,0.5068,0,Eastern Terai
2,2000-01-15,2000,15,Lumbini,Bardiya,Gulariya,28.23,81.33,25.32,9.21,...,0.2552,0.9669,0.5,0.866,Winter,0.0000,0.00,2.3437,1,Western Terai


In [3]:
df.info()
# df.columns

<class 'pandas.DataFrame'>
RangeIndex: 204582 entries, 0 to 204581
Data columns (total 65 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   Date                  204582 non-null  datetime64[ns]
 1   Year                  204582 non-null  int32         
 2   Day_of_Year           204582 non-null  int32         
 3   Province              204582 non-null  category      
 4   District              204582 non-null  category      
 5   Regional_Hub          204582 non-null  category      
 6   Latitude              204582 non-null  float64       
 7   Longitude             204582 non-null  float64       
 8   Max_Temp_C            204582 non-null  float64       
 9   Min_Temp_C            204582 non-null  float64       
 10  Mean_Temp_C           204582 non-null  float64       
 11  Humidity_Pct          204582 non-null  float64       
 12  Wind_Speed_ms         204582 non-null  float64       
 13  Precipitat

In [4]:
# target columns labels 
# y_next_day = df["Next_Day_Max_Temp"]
# y_heat_flag = df["Heatwave_Flag"]
# y_heat_level = df["Heatwave_Severity"]
y_next_day = df["Next_Day_Max_Temp"]
y_heat_flag = (df["Next_Day_Max_Temp"] >= 40.0).astype(int)
conditions = [
    df["Next_Day_Max_Temp"] < 40.0,
    (df["Next_Day_Max_Temp"] >= 40.0) & (df["Next_Day_Max_Temp"] < 42.0),
    (df["Next_Day_Max_Temp"] >= 42.0) & (df["Next_Day_Max_Temp"] < 45.0),
    df["Next_Day_Max_Temp"] >= 45.0
]
choices = [0, 1, 2, 3]
y_heat_level = pd.Series(np.select(conditions, choices, default=0), index=df.index, dtype="int8")



In [5]:
# decoupled
meta_cols = ["Date", "District", "Province", "Regional_Hub"]
df_meta = df[meta_cols].copy()

df_meta.sample(3)


,Date,District,Province,Regional_Hub
141802,2018-07-11,Mahottari,Madhesh,Jaleshwar
186036,2024-04-16,Sarlahi,Madhesh,Malangwa
203053,2026-07-06,Dang,Lumbini,Ghorahi


In [6]:
# encode
cat_cols = ["Season", "Terai_Region"]
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=False, dtype="int8")

dummy_cols = [c for c in df_encoded.columns if c.startswith("Season_") or c.startswith("Terai_Region_")]
df_encoded[dummy_cols].sample(3)


,Season_Monsoon,Season_Post-Monsoon,Season_Pre-Monsoon,Season_Winter,Terai_Region_Central Terai,Terai_Region_Eastern Terai,Terai_Region_Western Terai
187253,1,0,0,0,0,1,0
127051,1,0,0,0,0,1,0
88965,1,0,0,0,0,0,1


In [7]:
# 
target_cols = ["Next_Day_Max_Temp", "Heatwave_Flag", "Heatwave_Severity"]
drop_cols = meta_cols + target_cols
X = df_encoded.drop(columns=drop_cols)
X.sample(3)

,Year,Day_of_Year,Latitude,Longitude,Max_Temp_C,Min_Temp_C,Mean_Temp_C,Humidity_Pct,Wind_Speed_ms,Precipitation_mm,...,Yesterday_Rain_Log,Past_7D_Rain_Log,Is_Western_Terai,Season_Monsoon,Season_Post-Monsoon,Season_Pre-Monsoon,Season_Winter,Terai_Region_Central Terai,Terai_Region_Eastern Terai,Terai_Region_Western Terai
88254,2011,199,27.65,84.12,31.89,24.50,28.34,82.24,1.40,9.60,...,0.9821,3.6909,0,1,0,0,0,1,0,0
52867,2006,340,26.64,85.80,25.69,12.11,18.50,48.64,0.71,0.00,...,0.0000,0.0000,0,0,0,0,1,0,1,0
148785,2019,160,28.05,81.62,41.72,26.97,34.62,31.46,1.70,2.19,...,0.4574,3.5470,1,1,0,0,0,0,0,1


In [8]:
# train_val_test
train_mask = df_meta["Date"] <= "2020-12-31"
val_mask   = df_meta["Date"].between("2021-01-01", "2022-12-31")
test_mask  = df_meta["Date"] >= "2023-01-01"

X_train, X_val, X_test = X[train_mask], X[val_mask], X[test_mask]
y_train_flag, y_val_flag, y_test_flag = y_heat_flag[train_mask], y_heat_flag[val_mask], y_heat_flag[test_mask]
y_train_mtemp, y_val_mtemp, y_test_mtemp = y_next_day[train_mask], y_next_day[val_mask], y_next_day[test_mask]
y_train_sev,  y_val_sev,  y_test_sev  = y_heat_level[train_mask], y_heat_level[val_mask], y_heat_level[test_mask]


In [9]:
# fitting and saving
Path("../models").mkdir(parents=True, exist_ok=True)
scaler = StandardScaler()
scaler.fit(X_train)
joblib.dump(scaler, "../models/preprocessing.pkl")
print("✅ ../models/preprocessing.pkl saved")


✅ ../models/preprocessing.pkl saved


In [10]:
# scalling
X_train_scaled = scaler.transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_val_scaled shape  : {X_val_scaled.shape}")
print(f"X_test_scaled shape : {X_test_scaled.shape}")


X_train_scaled shape: (160797, 63)
X_val_scaled shape  : (15330, 63)
X_test_scaled shape : (28455, 63)


### Init models 


In [11]:
# ==============================================================================
# Unified Imports: Models, Metrics & Scoring
# ==============================================================================
import time
import numpy as np
import pandas as pd

# --- 1. Candidate Models by Family ---
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.linear_model import Lasso, LogisticRegression, Ridge
from xgboost import XGBClassifier, XGBRegressor

# --- 2. Classification Evaluation Metrics (Target 1 & Target 2) ---
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,  # PR-AUC
    brier_score_loss,  # Calibration
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# --- 3. Regression Evaluation Metrics (Target 3) ---
from sklearn.metrics import (
    max_error,
    mean_absolute_error,  # MAE (°C)
    mean_squared_error,  # RMSE (°C)
    r2_score,
)

# --- 4. Shared Negative-to-Positive Imbalance Ratio (~14.5:1) ---
pos_weight = (len(y_train_flag) - y_train_flag.sum()) / y_train_flag.sum()


In [12]:
models_target1_flag = {
    # 1. Linear Family (Requires scaled features, Cost-Sensitive)
    "Logistic Regression": LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=42
    ),
    # 2. Tree Bagging Family (Unscaled features, Parallelized)
    "Random Forest": RandomForestClassifier(
        n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=42
    ),
    # 3. Tree Boosting Family
    "HistGradientBoosting": HistGradientBoostingClassifier(
        class_weight="balanced", random_state=42
    ),
    "LightGBM": LGBMClassifier(
        scale_pos_weight=pos_weight,
        n_estimators=150,
        random_state=42,
        verbosity=-1,
        n_jobs=-1,
    ),
    "XGBoost": XGBClassifier(
        scale_pos_weight=pos_weight,
        n_estimators=150,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1,
    ),
}


models_target2_severity = {
    # 1. Linear Family (Automatically handles multinomial, Scaled)
    "Logistic Regression (Multinomial)": LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=42
    ),
    # 2. Tree Bagging Family (Unscaled features, Parallelized)
    "Random Forest": RandomForestClassifier(
        n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=42
    ),
    # 3. Tree Boosting Family (HistGradientBoosting Multi-Class)
    "HistGradientBoosting": HistGradientBoostingClassifier(
        class_weight="balanced", random_state=42
    ),
    # 4. Tree Boosting Family (LightGBM Multi-Class)
    "LightGBM": LGBMClassifier(
        objective="multiclass",
        num_class=4,
        class_weight="balanced",
        n_estimators=150,
        random_state=42,
        verbosity=-1,
        n_jobs=-1,
    ),
    # 5. Tree Boosting Family (XGBoost Multi-Class)
    "XGBoost": XGBClassifier(
        objective="multi:softprob",
        num_class=4,
        n_estimators=150,
        random_state=42,
        eval_metric="mlogloss",
        n_jobs=-1,
    ),
}

models_target3_temp = {
    # 1. Linear Family (L2 Regularized Baseline, Scaled)
    "Ridge Regression": Ridge(alpha=1.0, random_state=42),
    # 1. Linear Family (L1 Sparse Regularized, Scaled)
    "Lasso Regression": Lasso(alpha=0.1, random_state=42),
    # 2. Tree Bagging Family (Unscaled features, Parallelized)
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=100, n_jobs=-1, random_state=42
    ),
    # 3. Tree Boosting Family
    "HistGradientBoosting Regressor": HistGradientBoostingRegressor(
        random_state=42
    ),
    "LightGBM Regressor": LGBMRegressor(
        n_estimators=150, random_state=42, verbosity=-1, n_jobs=-1
    ),
    "XGBoost Regressor": XGBRegressor(
        n_estimators=150, random_state=42, n_jobs=-1
    ),
}




In [13]:
# Step 11: Train & Evaluate Target 1 Models

target1_results = []

for name, model in models_target1_flag.items():

    print(f"Training {name}...\n", end=" ")

    # Train
    model.fit(X_train_scaled, y_train_flag)

    # Predict
    val_preds = model.predict(X_val_scaled)
    val_probs = model.predict_proba(X_val_scaled)[:, 1]

    # Metrics
    precision = precision_score(y_val_flag, val_preds, zero_division=0)
    recall = recall_score(y_val_flag, val_preds, zero_division=0)
    f1 = f1_score(y_val_flag, val_preds, zero_division=0)
    pr_auc = average_precision_score(y_val_flag, val_probs)
    roc_auc = roc_auc_score(y_val_flag, val_probs)
    brier = brier_score_loss(y_val_flag, val_probs)

    # Save results
    target1_results.append({
        "Model": name,
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1": round(f1, 4),
        "PR-AUC": round(pr_auc, 4),
        "ROC-AUC": round(roc_auc, 4),
        "Brier": round(brier, 4)
    })

    # print(f"Recall={recall:.4f} | PR-AUC={pr_auc:.4f}")


# Leaderboard
df_target1_leaderboard = (
    pd.DataFrame(target1_results)
    .sort_values("PR-AUC", ascending=False)
    .reset_index(drop=True)
)

display(df_target1_leaderboard)

Training Logistic Regression...
 Training Random Forest...
 Training HistGradientBoosting...
 Training LightGBM...
 Training XGBoost...
 

,Model,Precision,Recall,F1,PR-AUC,ROC-AUC,Brier
0,Logistic Regression,0.3748,0.9722,0.5410,0.7657,0.9899,0.0354
1,HistGradientBoosting,0.5160,0.8590,0.6447,0.7462,0.9885,0.0207
2,Random Forest,0.6192,0.6880,0.6518,0.7360,0.9875,0.0151
3,LightGBM,0.5181,0.7970,0.6279,0.7249,0.9870,0.0201
4,XGBoost,0.5957,0.6453,0.6195,0.6853,0.9847,0.0181


In [14]:
# Step 12: Train & Evaluate Target 2 (Severity) Models

target2_results = []

for name, model in models_target2_severity.items():
    print(f"Training {name}...\n", end=" ")

    model.fit(X_train_scaled, y_train_sev)
    val_preds = model.predict(X_val_scaled)

    acc = accuracy_score(y_val_sev, val_preds)
    macro_f1 = f1_score(y_val_sev, val_preds, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_val_sev, val_preds, average="weighted", zero_division=0)

    target2_results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Macro F1": round(macro_f1, 4),
        "Weighted F1": round(weighted_f1, 4)
    })

    # print(f"Macro F1={macro_f1:.4f} | Accuracy={acc:.4f}")

df_target2_leaderboard = (
    pd.DataFrame(target2_results)
    .sort_values("Macro F1", ascending=False)
    .reset_index(drop=True)
)

display(df_target2_leaderboard)


Training Logistic Regression (Multinomial)...
 

e:\installexe\Anaconda\envs\DS\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Training Random Forest...
 Training HistGradientBoosting...
 Training LightGBM...
 Training XGBoost...
 

,Model,Accuracy,Macro F1,Weighted F1
0,XGBoost,0.9751,0.6076,0.9720
1,HistGradientBoosting,0.9451,0.5945,0.9579
2,LightGBM,0.9668,0.5848,0.9695
3,Random Forest,0.9705,0.5315,0.9730
4,Logistic Regression (Multinomial),0.9359,0.4798,0.9523


In [15]:
# Step 13: Train & Evaluate Target 3 (Temperature) Models

target3_results = []

for name, model in models_target3_temp.items():
    print(f"Training {name}...\n", end=" ")

    model.fit(X_train_scaled, y_train_mtemp)
    val_preds = model.predict(X_val_scaled)

    mae = mean_absolute_error(y_val_mtemp, val_preds)
    rmse = np.sqrt(mean_squared_error(y_val_mtemp, val_preds))
    r2 = r2_score(y_val_mtemp, val_preds)

    target3_results.append({
        "Model": name,
        "MAE (°C)": round(mae, 4),
        "RMSE (°C)": round(rmse, 4),
        "R²": round(r2, 4)
    })

    # print(f"MAE={mae:.2f}°C | R²={r2:.4f}")

df_target3_leaderboard = (
    pd.DataFrame(target3_results)
    .sort_values("MAE (°C)", ascending=True)
    .reset_index(drop=True)
)

display(df_target3_leaderboard)


Training Ridge Regression...
 Training Lasso Regression...
 Training Random Forest Regressor...
 Training HistGradientBoosting Regressor...
 Training LightGBM Regressor...
 Training XGBoost Regressor...
 

,Model,MAE (°C),RMSE (°C),R²
0,Ridge Regression,0.9482,1.2978,0.9487
1,Random Forest Regressor,0.9531,1.2969,0.9488
2,HistGradientBoosting Regressor,0.9677,1.3077,0.9479
3,LightGBM Regressor,0.9700,1.3087,0.9479
4,Lasso Regression,1.0072,1.3691,0.9429
5,XGBoost Regressor,1.0640,1.4390,0.9370


In [16]:
# Step 14A: Fine-Tune Target 1 Top-3 Models (Narrowed Decision Threshold Sweep)

from sklearn.metrics import (
    recall_score, precision_score, f1_score,
    accuracy_score, average_precision_score
)
import pandas as pd

top3_target1 = {
    "Logistic Regression": models_target1_flag["Logistic Regression"],
    "Random Forest": models_target1_flag["Random Forest"],
    "HistGradientBoosting": models_target1_flag["HistGradientBoosting"]
}

# Narrowed threshold ranges based on prior validation curves
focused_thresholds = {
    "Logistic Regression": [0.45, 0.50],
    "Random Forest": [0.25, 0.30],
    "HistGradientBoosting": [0.25, 0.30]
}

t1_tune_results = []

for name, model in top3_target1.items():
    probs = model.predict_proba(X_val_scaled)[:, 1]

    for t in focused_thresholds[name]:
        preds = (probs >= t).astype(int)

        t1_tune_results.append({
            "Model": name,
            "Threshold": t,
            "Recall (%)": round(recall_score(y_val_flag, preds) * 100, 2),
            "Precision (%)": round(precision_score(y_val_flag, preds, zero_division=0) * 100, 2),
            "F1-Score (%)": round(f1_score(y_val_flag, preds, zero_division=0) * 100, 2),
            "Accuracy (%)": round(accuracy_score(y_val_flag, preds) * 100, 2),
            "PR-AUC (%)": round(average_precision_score(y_val_flag, probs) * 100, 2),
            "Alerts": int(preds.sum())
        })

df_t1_tuned = (
    pd.DataFrame(t1_tune_results)
    .sort_values(by=["Recall (%)", "Precision (%)"], ascending=[False, False])
    .reset_index(drop=True)
)

display(df_t1_tuned)


,Model,Threshold,Recall (%),Precision (%),F1-Score (%),Accuracy (%),PR-AUC (%),Alerts
0,Logistic Regression,0.45,98.08,36.17,52.85,94.66,76.57,1269
1,Logistic Regression,0.50,97.22,37.48,54.10,94.96,76.57,1214
2,HistGradientBoosting,0.25,91.45,41.55,57.14,95.81,74.62,1030
3,HistGradientBoosting,0.30,90.17,43.87,59.02,96.18,74.62,962
4,Random Forest,0.25,88.89,46.90,61.40,96.59,73.60,887
5,Random Forest,0.30,85.26,50.96,63.79,97.05,73.60,783


In [17]:
# Step 14B: Hyperparameter Tuning — Target 2 Top-3 Models (Focused Grid)

from xgboost import XGBClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score, accuracy_score, recall_score
import pandas as pd

t2_configs = {}

# -----------------------------
# 1. LightGBM (Top Performer: leaves in [31, 63], lr in [0.05, 0.10])
# -----------------------------
for leaves in [31, 63]:
    for lr in [0.05, 0.10]:
        name = f"LightGBM (leaves={leaves}, lr={lr})"
        t2_configs[name] = LGBMClassifier(
            num_leaves=leaves,
            learning_rate=lr,
            n_estimators=200,
            class_weight="balanced",
            random_state=42,
            verbosity=-1,
            n_jobs=-1
        )

# -----------------------------
# 2. HistGradientBoosting (depth in [4, 6], lr in [0.05, 0.10])
# -----------------------------
for depth in [4, 6]:
    for lr in [0.05, 0.10]:
        name = f"HistGB (depth={depth}, lr={lr})"
        t2_configs[name] = HistGradientBoostingClassifier(
            max_iter=200,
            max_depth=depth,
            learning_rate=lr,
            class_weight="balanced",
            random_state=42
        )

# -----------------------------
# 3. XGBoost (depth in [4, 6], lr in [0.05, 0.10])
# -----------------------------
for depth in [4, 6]:
    for lr in [0.05, 0.10]:
        name = f"XGBoost (depth={depth}, lr={lr})"
        t2_configs[name] = XGBClassifier(
            max_depth=depth,
            learning_rate=lr,
            n_estimators=200,
            random_state=42,
            eval_metric="mlogloss",
            n_jobs=-1
        )

# -----------------------------
# Evaluate Focused Configurations
# -----------------------------
t2_tune_results = []

for name, model in t2_configs.items():
    model.fit(X_train_scaled, y_train_sev)
    preds = model.predict(X_val_scaled)
    recalls = recall_score(y_val_sev, preds, average=None, zero_division=0)

    t2_tune_results.append({
        "Model Config": name,
        "Macro F1 (%)": round(f1_score(y_val_sev, preds, average="macro", zero_division=0) * 100, 2),
        "Accuracy (%)": round(accuracy_score(y_val_sev, preds) * 100, 2),
        "Tier 2 Recall (%)": round((recalls[2] if len(recalls) > 2 else 0) * 100, 2),
        "Tier 3 Recall (%)": round((recalls[3] if len(recalls) > 3 else 0) * 100, 2)
    })

df_t2_tuned = (
    pd.DataFrame(t2_tune_results)
    .sort_values(by=["Macro F1 (%)", "Tier 3 Recall (%)"], ascending=[False, False])
    .reset_index(drop=True)
)

display(df_t2_tuned)


,Model Config,Macro F1 (%),Accuracy (%),Tier 2 Recall (%),Tier 3 Recall (%)
0,"LightGBM (leaves=63, lr=0.1)",66.50,97.21,79.57,50.0
1,"LightGBM (leaves=63, lr=0.05)",62.16,96.89,79.57,50.0
2,"LightGBM (leaves=31, lr=0.05)",59.82,96.31,77.42,50.0
3,"LightGBM (leaves=31, lr=0.1)",59.33,96.95,72.04,50.0
4,"HistGB (depth=6, lr=0.1)",55.63,95.34,70.97,50.0
5,"HistGB (depth=4, lr=0.1)",54.75,94.53,65.59,100.0
6,"HistGB (depth=6, lr=0.05)",53.17,94.14,68.82,50.0
7,"HistGB (depth=4, lr=0.05)",53.12,93.65,62.37,100.0
8,"XGBoost (depth=4, lr=0.05)",50.00,97.68,67.74,0.0
9,"XGBoost (depth=6, lr=0.1)",49.93,97.56,73.12,0.0


In [18]:
# Step 14C: Hyperparameter Tuning — Target 3 Top-3 Models (Focused Grid)

from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

t3_configs = {}

# -----------------------------
# 1. Ridge Regression (Best alphas: 0.1, 1.0)
# -----------------------------
for alpha in [0.1, 1.0]:
    name = f"Ridge (alpha={alpha})"
    t3_configs[name] = Ridge(alpha=alpha, random_state=42)

# -----------------------------
# 2. Random Forest Regressor (Best depths: 15, 20)
# -----------------------------
for depth in [15, 20]:
    name = f"RF (depth={depth})"
    t3_configs[name] = RandomForestRegressor(
        n_estimators=100,
        max_depth=depth,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )

# -----------------------------
# 3. HistGradientBoosting (Best depths: [6, 8], best lrs: [0.03, 0.05])
# -----------------------------
for depth in [6, 8]:
    for lr in [0.03, 0.05]:
        name = f"HistGB (depth={depth}, lr={lr})"
        t3_configs[name] = HistGradientBoostingRegressor(
            max_depth=depth,
            learning_rate=lr,
            max_iter=200,
            random_state=42
        )

# -----------------------------
# Evaluate Focused Configurations
# -----------------------------
t3_tune_results = []

for name, model in t3_configs.items():
    model.fit(X_train_scaled, y_train_mtemp)
    preds = model.predict(X_val_scaled)

    t3_tune_results.append({
        "Model Config": name,
        "MAE (°C)": round(mean_absolute_error(y_val_mtemp, preds), 4),
        "RMSE (°C)": round(np.sqrt(mean_squared_error(y_val_mtemp, preds)), 4),
        "R² Score": round(r2_score(y_val_mtemp, preds), 4)
    })

df_t3_tuned = (
    pd.DataFrame(t3_tune_results)
    .sort_values(by=["MAE (°C)", "RMSE (°C)"], ascending=[True, True])
    .reset_index(drop=True)
)

display(df_t3_tuned)


,Model Config,MAE (°C),RMSE (°C),R² Score
0,RF (depth=15),0.9479,1.2938,0.9490
1,Ridge (alpha=0.1),0.9482,1.2978,0.9487
2,Ridge (alpha=1.0),0.9482,1.2978,0.9487
3,RF (depth=20),0.9495,1.2940,0.9490
4,"HistGB (depth=6, lr=0.03)",0.9553,1.2942,0.9490
5,"HistGB (depth=8, lr=0.03)",0.9581,1.2968,0.9488
6,"HistGB (depth=8, lr=0.05)",0.9603,1.2978,0.9487
7,"HistGB (depth=6, lr=0.05)",0.9645,1.3009,0.9485


In [19]:
# Step 15B & 16: Set Optimal Threshold & Crown Champions

# 1. Target 1: Set operational threshold (85.26% recall, 50.96% precision)
OPTIMAL_THRESHOLD_FLAG = 0.30
val_probs = models_target1_flag["Random Forest"].predict_proba(X_val_scaled)[:, 1]
optimal_val_preds_flag = (val_probs >= OPTIMAL_THRESHOLD_FLAG).astype(int)

# 2. Crown the operational champions based on condition & target requirements:
champion_flag     = models_target1_flag["Random Forest"]
champion_severity = t2_configs["LightGBM (leaves=63, lr=0.1)"]
champion_temp     = t3_configs["Ridge (alpha=1.0)"]  # Note: Use t3_configs["RF (depth=15)"] if you prefer Random Forest

print("Operational Champions Crowned:")
print(f"  Target 1 (Alert):       Random Forest Classifier @ Decision Threshold = {OPTIMAL_THRESHOLD_FLAG}")
print(f"  Target 2 (Severity):    LightGBM Classifier (leaves=63, lr=0.10, class_weight='balanced')")
print(f"  Target 3 (Temperature): Ridge Regression (alpha=1.0)")


Operational Champions Crowned:
  Target 1 (Alert):       Random Forest Classifier @ Decision Threshold = 0.3
  Target 2 (Severity):    LightGBM Classifier (leaves=63, lr=0.10, class_weight='balanced')
  Target 3 (Temperature): Ridge Regression (alpha=1.0)


# Multi-Target Consistency & Explainability

-  Enforce prediction consistency and audit contradictions.
- Analyze feature importance.
- Final: Test the consistency engine with a sample prediction.

In [20]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    recall_score,
    precision_score,
    f1_score,
    accuracy_score,
    mean_absolute_error,
    r2_score
)

##  Multi-Target Consistency Enforcer

Align heatwave flags, severity levels, and predicted temperatures
so that the final predictions follow the defined rules.

In [21]:
# Train the Conditioned Specialist for Target 3 Re-prediction
# Features: [Weather Scaled (62 cols) + Flag (1 col) + Severity (1 col)]
X_train_cond = np.column_stack([X_train_scaled, y_train_flag, y_train_sev])
conditioned_temp_model = Ridge(alpha=1.0, random_state=42)
conditioned_temp_model.fit(X_train_cond, y_train_mtemp)


,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary <random_state>` for details... versionadded:: 0.17 `random_state` to support Stochastic Average Gradient.",42
,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"pos

In [22]:
def batch_enforce_consistency_hybrid(flags, severities, temps, X_scaled, conditioned_model):
    clean_flags = flags.copy().astype(int)
    clean_sevs = severities.copy().astype(int)
    clean_temps = temps.copy().astype(float)

    # 1. Safety Promotion: If Severity and Temp both see a heatwave, fix Flag to 1!
    missed_heatwave_mask = (clean_flags == 0) & (clean_sevs >= 1) & (clean_temps >= 40.0)
    clean_flags[missed_heatwave_mask] = 1

    # 2. Demote Severity if Flag == 0
    clean_sevs[clean_flags == 0] = 0

    # 3. Smart Severity Healing: If Flag == 1 but Severity == 0, infer Severity directly from Temp!
    missed_sev_mask = (clean_flags == 1) & (clean_sevs == 0)
    if missed_sev_mask.sum() > 0:
        temp_tiers = np.where(clean_temps >= 45.0, 3,
                     np.where(clean_temps >= 42.0, 2,
                     np.where(clean_temps >= 40.0, 1, 1)))
        clean_sevs[missed_sev_mask] = temp_tiers[missed_sev_mask]

    # 4. Target 3 Conflict Check (Does temp match the resolved severity tier?)
    c0 = (clean_sevs == 0) & (clean_temps >= 40.0)
    c1 = (clean_sevs == 1) & ~((clean_temps >= 40.0) & (clean_temps < 42.0))
    c2 = (clean_sevs == 2) & ~((clean_temps >= 42.0) & (clean_temps < 45.0))
    c3 = (clean_sevs == 3) & (clean_temps < 45.0)
    conflict_mask = c0 | c1 | c2 | c3

    # 5. Only re-predict Target 3 when a conflict occurs!
    if conflict_mask.sum() > 0:
        X_cond_conflicts = np.column_stack([
            X_scaled[conflict_mask],
            clean_flags[conflict_mask],
            clean_sevs[conflict_mask]
        ])
        repredicted_temps = conditioned_model.predict(X_cond_conflicts)

        for i, idx in enumerate(np.where(conflict_mask)[0]):
            sev = clean_sevs[idx]
            t = repredicted_temps[i]
            if sev == 0 and t >= 40.0:
                t = 39.9
            elif sev == 1:
                t = float(np.clip(t, 40.0, 41.9))
            elif sev == 2:
                t = float(np.clip(t, 42.0, 44.9))
            elif sev == 3 and t < 45.0:
                t = 45.0
            clean_temps[idx] = t

    return clean_flags, clean_sevs, clean_temps


## Validation Consistency Audit

Generate raw predictions, enforce consistency, and compare logical
contradictions before and after enforcement.

In [23]:
# Use the same preprocessing used when each model was trained.
raw_val_flags = (
    champion_flag.predict_proba(X_val_scaled)[:, 1] >= OPTIMAL_THRESHOLD_FLAG
).astype(int)

raw_val_sevs = champion_severity.predict(X_val_scaled)
raw_val_temps = champion_temp.predict(X_val_scaled)

clean_val_flags, clean_val_sevs, clean_val_temps = (
    batch_enforce_consistency_hybrid(
        raw_val_flags,
        raw_val_sevs,
        raw_val_temps,
        X_val_scaled,
        conditioned_temp_model,
    )
)


In [24]:
def audit_contradictions(flags, severities, temps):
    checks = {
        "Flag=0 & Severity>0": (flags == 0) & (severities > 0),
        "Flag=1 & Severity=0": (flags == 1) & (severities == 0),
        "Severity=0 & Temp>=40°C": (severities == 0) & (temps >= 40),
        "Severity>0 & Temp<40°C": (severities > 0) & (temps < 40)
    }

    results = {name: int(mask.sum()) for name, mask in checks.items()}
    results["Total Contradictions"] = int(
        np.logical_or.reduce(list(checks.values())).sum()
    )

    return results


raw_audit = audit_contradictions(
    raw_val_flags, raw_val_sevs, raw_val_temps
)

clean_audit = audit_contradictions(
    clean_val_flags, clean_val_sevs, clean_val_temps
)

df_audit = pd.DataFrame({
    "Before Enforcement": raw_audit,
    "After Enforcement": clean_audit
})

df_audit["Reduction (%)"] = np.where(
    df_audit["Before Enforcement"] > 0,
    (
        (df_audit["Before Enforcement"] - df_audit["After Enforcement"])
        / df_audit["Before Enforcement"] * 100
    ).round(2),
    0.0
)

display(df_audit)

,Before Enforcement,After Enforcement,Reduction (%)
Flag=0 & Severity>0,11,0,100.0
Flag=1 & Severity=0,346,0,100.0
Severity=0 & Temp>=40°C,68,0,100.0
Severity>0 & Temp<40°C,133,0,100.0
Total Contradictions,479,0,100.0


In [25]:
metric_comparison = pd.DataFrame([
    {
        "Target": "Heatwave Alert",
        "Metric": "Recall (%)",
        "Before": recall_score(y_val_flag, raw_val_flags) * 100,
        "After": recall_score(y_val_flag, clean_val_flags) * 100
    },
    {
        "Target": "Heatwave Alert",
        "Metric": "Accuracy (%)",
        "Before": accuracy_score(y_val_flag, raw_val_flags) * 100,
        "After": accuracy_score(y_val_flag, clean_val_flags) * 100
    },
    {
        "Target": "Severity",
        "Metric": "Macro F1 (%)",
        "Before": f1_score(y_val_sev, raw_val_sevs, average="macro") * 100,
        "After": f1_score(y_val_sev, clean_val_sevs, average="macro") * 100
    },
    {
        "Target": "Maximum Temperature",
        "Metric": "MAE (°C)",
        "Before": mean_absolute_error(y_val_mtemp, raw_val_temps),
        "After": mean_absolute_error(y_val_mtemp, clean_val_temps)
    },
    {
        "Target": "Maximum Temperature",
        "Metric": "R² (%)",
        "Before": r2_score(y_val_mtemp, raw_val_temps) * 100,
        "After": r2_score(y_val_mtemp, clean_val_temps) * 100
    }
])

metric_comparison[["Before", "After"]] = (
    metric_comparison[["Before", "After"]].round(2)
)

display(metric_comparison)

,Target,Metric,Before,After
0,Heatwave Alert,Recall (%),85.26,85.26
1,Heatwave Alert,Accuracy (%),97.05,97.05
2,Severity,Macro F1 (%),66.50,68.90
3,Maximum Temperature,MAE (°C),0.95,0.96
4,Maximum Temperature,R² (%),94.87,94.75


##  Meteorological Feature Importance

Identify the features most influential to each champion model.
Ridge coefficients show direction and magnitude; tree importances
show relative importance, not causation.

In [26]:
rf_fi = pd.Series(
    champion_flag.feature_importances_, index=X.columns
).sort_values(ascending=False).head(10)

# Normalize LightGBM split counts to sum to 1.0 (true percentage)
lgbm_raw = champion_severity.feature_importances_
lgbm_normalized = lgbm_raw / lgbm_raw.sum() if lgbm_raw.sum() > 0 else lgbm_raw
lgbm_fi = pd.Series(lgbm_normalized, index=X.columns).sort_values(
    ascending=False
).head(10)

ridge_coef = pd.Series(champion_temp.coef_, index=X.columns).sort_values(
    key=abs, ascending=False
).head(10)

df_fi_summary = pd.DataFrame({
    "Heatwave Alert (Random Forest)": [
        f"{name} ({value * 100:.1f}%)" for name, value in rf_fi.items()
    ],
    "Severity (LightGBM)": [
        f"{name} ({value * 100:.1f}%)" for name, value in lgbm_fi.items()
    ],
    "Temperature (Ridge)": [
        f"{name} ({value:+.2f})" for name, value in ridge_coef.items()
    ],
})

display(df_fi_summary)


,Heatwave Alert (Random Forest),Severity (LightGBM),Temperature (Ridge)
0,Max_Temp_C (14.8%),Wind_Speed_ms (3.6%),Mean_Temp_C (+2.67)
1,Yesterday_Max_Temp (9.8%),Night_Momentum_2D (3.5%),Day_Thermal_Gap (-1.97)
2,Mean_Temp_C (9.2%),Temp_Momentum_1D (3.5%),Wet_Bulb_Temp (+1.33)
3,Past_3D_Avg_Max_Temp (8.1%),Max_Temp_C (3.4%),Temp_Range (+1.04)
4,Max_Temp_Lag2 (6.0%),Temp_Momentum_2D (3.4%),Max_Temp_C (+0.93)
5,Past_7D_Max_Temp (5.8%),Night_Momentum_1D (3.1%),Night_Thermal_Gap (+0.80)
6,Clim_Max_Mean (5.2%),Past_14D_Rain_Sum (3.1%),Past_7D_Avg_Max_Temp (+0.62)
7,Loo_Index (4.0%),Clim_Min_Std (3.1%),Rain_Log (-0.54)
8,Heatwave_Streak (3.7%),Clim_Max_Std (3.0%),Past_3D_Avg_Max_Temp (+0.54)
9,Wet_Bulb_Depression (3.5%),Temp_Range (2.7%),Feels_Like_Min_C (+0.43)


##  Single Prediction

Test whether the consistency enforcer corrects a deliberately
contradictory set of model predictions.

In [27]:
def enforce_consistency_single(
    flag, severity, temp, x_row=None, conditioned_model=conditioned_temp_model
):
    if x_row is None:
        # Default baseline: validation average feature vector
        x_row = X_val_scaled.mean(axis=0)

    # Ensure compatibility with pd.Series, list, or np.ndarray
    x_arr = np.asarray(x_row).reshape(1, -1)

    f, s, t = batch_enforce_consistency_hybrid(
        np.array([flag]),
        np.array([severity]),
        np.array([temp]),
        x_arr,
        conditioned_model,
    )

    return {
        "heatwave_flag": int(f[0]),
        "heatwave_severity": int(s[0]),
        "predicted_max_temp": round(float(t[0]), 2),
        "raw_temp": round(float(temp), 2),
        "adjusted": (
            int(f[0]) != flag
            or int(s[0]) != severity
            or round(float(t[0]), 2) != round(float(temp), 2)
        ),
    }

# --- Demonstration & Verification ---
print("--- Case 1: Flag=0 (No Alert), Severity=2 (Moderate), Temp=43.5°C ---")
res1 = enforce_consistency_single(flag=0, severity=2, temp=43.5)
print("Output:", res1)

print("\n--- Case 2: Flag=1 (Alert), Severity=1 (Minor), Temp=38.2°C (<40°C conflict) ---")
res2 = enforce_consistency_single(flag=1, severity=1, temp=38.2)
print("Output:", res2)


--- Case 1: Flag=0 (No Alert), Severity=2 (Moderate), Temp=43.5°C ---
Output: {'heatwave_flag': 1, 'heatwave_severity': 2, 'predicted_max_temp': 43.5, 'raw_temp': 43.5, 'adjusted': True}

--- Case 2: Flag=1 (Alert), Severity=1 (Minor), Temp=38.2°C (<40°C conflict) ---
Output: {'heatwave_flag': 1, 'heatwave_severity': 1, 'predicted_max_temp': 40.0, 'raw_temp': 38.2, 'adjusted': True}


# 6. Final Evaluation on Held-Out Test Set (2023–2026)

Evaluate the crowned champion models on the completely untouched real-world test partition (2023–2026, $N=28,455$).
We will:
1. Generate raw model predictions on `X_test_scaled`.
2. Apply the Hybrid Multi-Target Consistency Enforcer.
3. Audit physical contradictions before and after enforcement on the test set.
4. Calculate the official final test performance scorecard for all 3 targets.
5. Serialize the champion models and metadata to `../models/`.


In [28]:
# 1. Generate raw predictions from the Champion Models on Test Set (2023–2026)
raw_test_flags = (
    champion_flag.predict_proba(X_test_scaled)[:, 1] >= OPTIMAL_THRESHOLD_FLAG
).astype(int)

raw_test_sevs = champion_severity.predict(X_test_scaled).astype(int)
raw_test_temps = champion_temp.predict(X_test_scaled).astype(float)

# 2. Apply the Hybrid Multi-Target Consistency Enforcer
clean_test_flags, clean_test_sevs, clean_test_temps = (
    batch_enforce_consistency_hybrid(
        raw_test_flags,
        raw_test_sevs,
        raw_test_temps,
        X_test_scaled,
        conditioned_temp_model,
    )
)

print(f"Total Test Records Evaluated: {len(X_test_scaled):,}")
print("Raw test predictions and consistency enforcement completed successfully!")


Total Test Records Evaluated: 28,455
Raw test predictions and consistency enforcement completed successfully!


In [29]:
# Audit physical contradictions on the Test Set before vs. after enforcement
raw_test_audit = audit_contradictions(
    raw_test_flags, raw_test_sevs, raw_test_temps
)
clean_test_audit = audit_contradictions(
    clean_test_flags, clean_test_sevs, clean_test_temps
)

df_test_audit = pd.DataFrame(
    {"Before Enforcement": raw_test_audit, "After Enforcement": clean_test_audit}
)

df_test_audit["Reduction (%)"] = np.where(
    df_test_audit["Before Enforcement"] > 0,
    (
        (
            df_test_audit["Before Enforcement"]
            - df_test_audit["After Enforcement"]
        )
        / df_test_audit["Before Enforcement"]
        * 100
    ).round(2),
    0.0,
)

print("=== Physical Contradiction Audit on Test Set (2023–2026) ===")
display(df_test_audit)


=== Physical Contradiction Audit on Test Set (2023–2026) ===


,Before Enforcement,After Enforcement,Reduction (%)
Flag=0 & Severity>0,37,0,100.0
Flag=1 & Severity=0,715,0,100.0
Severity=0 & Temp>=40°C,120,0,100.0
Severity>0 & Temp<40°C,523,0,100.0
Total Contradictions,1240,0,100.0


In [30]:
# Official Final Test Performance Scorecard
test_metric_comparison = pd.DataFrame([
    {
        "Target": "Target 1: Heatwave Alert",
        "Metric": "Recall (%)",
        "Before Enforcement": (
            recall_score(y_test_flag, raw_test_flags, zero_division=0) * 100
        ),
        "After Enforcement": (
            recall_score(y_test_flag, clean_test_flags, zero_division=0) * 100
        ),
    },
    {
        "Target": "Target 1: Heatwave Alert",
        "Metric": "Precision (%)",
        "Before Enforcement": (
            precision_score(y_test_flag, raw_test_flags, zero_division=0) * 100
        ),
        "After Enforcement": (
            precision_score(y_test_flag, clean_test_flags, zero_division=0) * 100
        ),
    },
    {
        "Target": "Target 1: Heatwave Alert",
        "Metric": "Accuracy (%)",
        "Before Enforcement": (
            accuracy_score(y_test_flag, raw_test_flags) * 100
        ),
        "After Enforcement": (
            accuracy_score(y_test_flag, clean_test_flags) * 100
        ),
    },
    {
        "Target": "Target 2: Severity Level",
        "Metric": "Macro F1 (%)",
        "Before Enforcement": (
            f1_score(y_test_sev, raw_test_sevs, average="macro", zero_division=0)
            * 100
        ),
        "After Enforcement": (
            f1_score(
                y_test_sev, clean_test_sevs, average="macro", zero_division=0
            )
            * 100
        ),
    },
    {
        "Target": "Target 2: Severity Level",
        "Metric": "Accuracy (%)",
        "Before Enforcement": (accuracy_score(y_test_sev, raw_test_sevs) * 100),
        "After Enforcement": (
            accuracy_score(y_test_sev, clean_test_sevs) * 100
        ),
    },
    {
        "Target": "Target 3: Maximum Temperature",
        "Metric": "MAE (°C)",
        "Before Enforcement": mean_absolute_error(y_test_mtemp, raw_test_temps),
        "After Enforcement": mean_absolute_error(
            y_test_mtemp, clean_test_temps
        ),
    },
    {
        "Target": "Target 3: Maximum Temperature",
        "Metric": "RMSE (°C)",
        "Before Enforcement": np.sqrt(
            mean_squared_error(y_test_mtemp, raw_test_temps)
        ),
        "After Enforcement": np.sqrt(
            mean_squared_error(y_test_mtemp, clean_test_temps)
        ),
    },
    {
        "Target": "Target 3: Maximum Temperature",
        "Metric": "R² Score (%)",
        "Before Enforcement": (
            r2_score(y_test_mtemp, raw_test_temps) * 100
        ),
        "After Enforcement": (
            r2_score(y_test_mtemp, clean_test_temps) * 100
        ),
    },
])

test_metric_comparison[["Before Enforcement", "After Enforcement"]] = (
    test_metric_comparison[["Before Enforcement", "After Enforcement"]].round(2)
)

print(
    "=== Official Final Test Performance Scorecard (2023–2026, N=28,455) ==="
)
display(test_metric_comparison)


=== Official Final Test Performance Scorecard (2023–2026, N=28,455) ===


,Target,Metric,Before Enforcement,After Enforcement
0,Target 1: Heatwave Alert,Recall (%),90.13,90.13
1,Target 1: Heatwave Alert,Precision (%),64.14,64.14
2,Target 1: Heatwave Alert,Accuracy (%),96.18,96.18
3,Target 2: Severity Level,Macro F1 (%),65.31,66.51
4,Target 2: Severity Level,Accuracy (%),95.25,94.62
5,Target 3: Maximum Temperature,MAE (°C),1.02,1.03
6,Target 3: Maximum Temperature,RMSE (°C),1.42,1.44
7,Target 3: Maximum Temperature,R² Score (%),93.29,93.14


In [31]:
# ==============================================================================
# Final Operational Deployment: Full Retraining on 2000–2026 Data (N=204,582)
# ==============================================================================
import json
import time
from pathlib import Path
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import Ridge

print("=" * 80)
print(">>> TRAINING FINAL PRODUCTION MODELS ON FULL 2000–2026 DATASET")
print(f"Total Operational Training Records: {len(X):,} daily district records")
print("=" * 80)

t_start = time.time()

# 1. Fit Final Full-History Scaler
scaler_full = StandardScaler()
X_full_scaled = scaler_full.fit_transform(X)

# 2. Retrain Target 1 Champion (Random Forest @ 100 trees, balanced weights)
print("1/4 Retraining Target 1 Alert Champion (Random Forest)...")
champion_flag_prod = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
champion_flag_prod.fit(X_full_scaled, y_heat_flag)

# 3. Retrain Target 2 Champion (LightGBM @ 63 leaves, balanced weights)
print("2/4 Retraining Target 2 Severity Champion (LightGBM)...")
champion_sev_prod = LGBMClassifier(
    num_leaves=63,
    learning_rate=0.10,
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
champion_sev_prod.fit(X_full_scaled, y_heat_level)

# 4. Retrain Target 3 Champion (Ridge Regression alpha=1.0)
print("3/4 Retraining Target 3 Temperature Champion (Ridge)...")
champion_temp_prod = Ridge(alpha=1.0, random_state=42)
champion_temp_prod.fit(X_full_scaled, y_next_day)

# 5. Retrain Secondary Specialist Regressor [Weather + Alert + Severity]
print("4/4 Retraining Conditioned Specialist Regressor (Ridge)...")
X_full_cond = np.column_stack([X_full_scaled, y_heat_flag, y_heat_level])
conditioned_model_prod = Ridge(alpha=1.0, random_state=42)
conditioned_model_prod.fit(X_full_cond, y_next_day)

train_duration = time.time() - t_start
print(f"✅ All production models successfully retrained in {train_duration:.2f} seconds!")

# ==============================================================================
# Serialize Artifacts to models/
# ==============================================================================
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(scaler_full, MODELS_DIR / "preprocessing.pkl")
joblib.dump(champion_flag_prod, MODELS_DIR / "heatwave_flag.pkl")
joblib.dump(champion_sev_prod, MODELS_DIR / "heatwave_severity.pkl")
joblib.dump(champion_temp_prod, MODELS_DIR / "heatwave_temp.pkl")
joblib.dump(conditioned_model_prod, MODELS_DIR / "heatwave_repredict_temp_model.pkl")

metadata = {
    "system_name": "Nepal Terai Extreme Heatwave Early-Warning System",
    "optimal_threshold_flag": OPTIMAL_THRESHOLD_FLAG,
    "n_features": len(X.columns),
    "feature_names": X.columns.tolist(),
    "training_scope": "Full Dataset Retrained (2000-01-15 to 2026-09-16, N=204,582 daily district records)",
    "champion_models": {
        "target_1_alert": "RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)",
        "target_2_severity": "LGBMClassifier(num_leaves=63, learning_rate=0.10, n_estimators=200, class_weight='balanced', random_state=42)",
        "target_3_temperature": "Ridge(alpha=1.0, random_state=42)",
        "conditioned_specialist": "Ridge(alpha=1.0, random_state=42) on [Weather + Alert + Severity]",
    },
    "consistency_enforcer": "Hybrid Multi-Target Consistency Enforcer (Logical Cascade + Conditioned Specialist Re-prediction)",
    "exported_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}

with open(MODELS_DIR / "pipeline_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

print(f"📦 Production artifacts exported to: {MODELS_DIR.resolve()}")


>>> TRAINING FINAL PRODUCTION MODELS ON FULL 2000–2026 DATASET
Total Operational Training Records: 204,582 daily district records
1/4 Retraining Target 1 Alert Champion (Random Forest)...
2/4 Retraining Target 2 Severity Champion (LightGBM)...
3/4 Retraining Target 3 Temperature Champion (Ridge)...
4/4 Retraining Conditioned Specialist Regressor (Ridge)...
✅ All production models successfully retrained in 14.11 seconds!
📦 Production artifacts exported to: E:\ALL_CODES_HERE_NOW\Project_Prot\VS\terai_heatwave_dataset\models
